# Simple RAG Pipeline - Complete Flow

This notebook demonstrates the complete RAG (Retrieval-Augmented Generation) pipeline in a simple, single-file flow without using classes.

## Pipeline Steps:
1. **Load Documents** - Load text from files
2. **Split into Chunks** - Break documents into smaller pieces
3. **Create Embeddings** - Convert text to numerical vectors
4. **Build FAISS Index** - Store embeddings for fast search
5. **Search** - Find relevant documents for a query
6. **Generate Answer** - Use retrieved context with LLM


In [2]:
# Step 1: Import all required libraries
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from pathlib import Path
import os

print("✓ All imports successful!")


✓ All imports successful!


## Step 2: Load Documents

First, let's create some sample documents or load from files.


In [3]:
# Option 1: Create sample documents directly (for demo)
from langchain_core.documents import Document

# Sample documents - in real use, you'd load from files
documents = [
    Document(
        page_content="Our company offers a 30-day money-back guarantee. If you're not satisfied with your purchase, you can return it within 30 days for a full refund. Contact support@company.com for returns.",
        metadata={"source": "refund_policy.txt", "topic": "refund"}
    ),
    Document(
        page_content="To configure API authentication, you need to generate an API key from the dashboard. Go to Settings > API Keys > Generate New Key. Store this key securely and use it in the Authorization header.",
        metadata={"source": "api_guide.txt", "topic": "api"}
    ),
    Document(
        page_content="The system requires Python 3.8 or higher, 4GB RAM minimum, and 10GB free disk space. Supported operating systems include Windows 10+, macOS 10.15+, and Linux Ubuntu 18.04+.",
        metadata={"source": "requirements.txt", "topic": "system"}
    ),
    Document(
        page_content="Our customer support is available 24/7 via email and chat. For urgent issues, call our hotline at 1-800-SUPPORT. Average response time is under 2 hours.",
        metadata={"source": "support_info.txt", "topic": "support"}
    ),
]

print(f"✓ Loaded {len(documents)} documents")
for i, doc in enumerate(documents, 1):
    print(f"  Document {i}: {doc.metadata['source']} ({len(doc.page_content)} chars)")


✓ Loaded 4 documents
  Document 1: refund_policy.txt (186 chars)
  Document 2: api_guide.txt (194 chars)
  Document 3: requirements.txt (172 chars)
  Document 4: support_info.txt (152 chars)


### Alternative: Load from Files

If you have actual files, uncomment and use this instead:


In [ ]:
# Option 2: Load from actual files (uncomment to use)
# documents_dir = Path("rag/documents")
# documents = []
# 
# if documents_dir.exists():
#     for file_path in documents_dir.glob("*.txt"):
#         loader = TextLoader(str(file_path), encoding="utf-8")
#         docs = loader.load()
#         for doc in docs:
#             doc.metadata["source"] = str(file_path)
#         documents.extend(docs)
#     print(f"✓ Loaded {len(documents)} documents from files")
# else:
#     print("⚠ Documents directory not found, using sample documents")


## Step 3: Split Documents into Chunks

Large documents need to be split into smaller chunks for better retrieval.


In [6]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,      # Size of each chunk
    chunk_overlap=50,    # Overlap between chunks (for context)
    length_function=len
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"✓ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"  Content: {chunks[0].page_content[:100]}...")
print(f"  Source: {chunks[0].metadata.get('source', 'N/A')}")


✓ Split 4 documents into 4 chunks

Sample chunk:
  Content: Our company offers a 30-day money-back guarantee. If you're not satisfied with your purchase, you ca...
  Source: refund_policy.txt


## Step 4: Create Embeddings

Convert text chunks into numerical vectors (embeddings) that capture semantic meaning.


In [ ]:
# Initialize embedding model
# This downloads the model on first run (may take a minute)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✓ Embedding model loaded")
print(f"  Model: sentence-transformers/all-MiniLM-L6-v2")
print(f"  Embedding dimension: 384")

# Test embedding (optional - just to see what it looks like)
sample_embedding = embeddings.embed_query("Hello world")
print(f"  Sample embedding shape: {len(sample_embedding)} dimensions")



d:\Work\Projects\LangChain\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 125.83it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Embedding model loaded
  Model: sentence-transformers/all-MiniLM-L6-v2
  Embedding dimension: 384
  Sample embedding shape: 384 dimensions


In [10]:
print(sample_embedding)

[-0.03447723388671875, 0.031023165211081505, 0.006734973751008511, 0.026108991354703903, -0.039362046867609024, -0.16030244529247284, 0.06692394614219666, -0.0064415112137794495, -0.04745052382349968, 0.014758817851543427, 0.07087530195713043, 0.05552754923701286, 0.019193369895219803, -0.026251282542943954, -0.01010951679199934, -0.02694047801196575, 0.022307446226477623, -0.022226616740226746, -0.1496925950050354, -0.017493078485131264, 0.007676276378333569, 0.054352231323719025, 0.0032544422429054976, 0.03172587975859642, -0.0846213772892952, -0.029405953362584114, 0.05159562826156616, 0.04812408611178398, -0.0033148485235869884, -0.05827920883893967, 0.04196930304169655, 0.022210706025362015, 0.128188818693161, -0.02233896777033806, -0.011656241491436958, 0.06292831897735596, -0.032876331359148026, -0.09122606366872787, -0.031175393611192703, 0.05269952118396759, 0.047034814953804016, -0.0842030942440033, -0.030056195333600044, -0.020744822919368744, 0.009517833590507507, -0.003721

## Step 5: Build FAISS Vector Store

Create a FAISS index from the document chunks. This enables fast similarity search.


In [14]:
# Create FAISS vector store from chunks
# This automatically:
# 1. Converts all chunks to embeddings
# 2. Stores them in a FAISS index
# 3. Enables fast similarity search
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("✓ FAISS vector store created")
print(f"  Total chunks indexed: {vector_store.index.ntotal if hasattr(vector_store.index, 'ntotal') else 'N/A'}")


ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

## Step 6: Save Vector Store (Optional)

Save the index to disk so you don't have to rebuild it every time.


In [ ]:
# Save vector store to disk
vector_store_path = "rag/vector_store/simple_faiss_index"
os.makedirs(os.path.dirname(vector_store_path), exist_ok=True)

vector_store.save_local(
    vector_store_path,
    allow_dangerous_deserialization=True
)

print(f"✓ Vector store saved to: {vector_store_path}")

# To load it later:
# vector_store = FAISS.load_local(
#     vector_store_path,
#     embeddings,
#     allow_dangerous_deserialization=True
# )


## Step 7: Search for Relevant Documents

Now we can search the vector store to find documents relevant to a query.


In [ ]:
# Define a search query
query = "What is the refund policy?"

# Search for relevant documents
# This automatically:
# 1. Converts query to embedding
# 2. Finds most similar document chunks
# 3. Returns top-k results

top_k = 2  # Number of results to retrieve
results = vector_store.similarity_search(query, k=top_k)

print(f"Query: '{query}'")
print(f"\n✓ Found {len(results)} relevant documents:\n")

for i, doc in enumerate(results, 1):
    print(f"--- Result {i} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content: {doc.page_content}")
    print()


## Step 8: Format Context for LLM

Combine retrieved documents into a context string for the LLM.


In [ ]:
# Format retrieved documents as context
def format_context(documents):
    """Format retrieved documents into a context string."""
    context_parts = []
    for i, doc in enumerate(documents, 1):
        source = doc.metadata.get('source', 'Unknown')
        content = doc.page_content
        context_parts.append(f"[Document {i} from {source}]\n{content}")
    return "\n\n---\n\n".join(context_parts)

# Get context for our query
context = format_context(results)

print("Formatted Context:")
print("=" * 60)
print(context)
print("=" * 60)


## Step 9: Generate Answer with LLM

Use the retrieved context along with the query to generate an answer.


In [ ]:
# Initialize LLM (using Groq - you can use any LLM)
# Note: You'll need GROQ_API_KEY in your .env file
from config.settings import settings

llm = ChatGroq(
    model=settings.MODEL_NAME,
    api_key=settings.GROQ_API_KEY,
    temperature=0.0
)

# Create a prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the provided context to answer questions accurately. If the context doesn't contain the answer, say so."),
    ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
])

# Generate answer
response = llm.invoke(prompt.format(context=context, question=query))

print(f"Question: {query}\n")
print(f"Answer: {response.content}")


## Step 10: Complete RAG Function

Let's create a simple function that does everything in one go:


In [ ]:
def simple_rag_search(query, vector_store, llm, top_k=2):
    """
    Simple RAG search function - does everything in one go.
    
    Args:
        query: User's question
        vector_store: FAISS vector store
        llm: Language model
        top_k: Number of documents to retrieve
    
    Returns:
        Answer string
    """
    # 1. Search for relevant documents
    results = vector_store.similarity_search(query, k=top_k)
    
    # 2. Format context
    context = format_context(results)
    
    # 3. Create prompt
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Use the provided context to answer questions accurately."),
        ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
    ])
    
    # 4. Generate answer
    response = llm.invoke(prompt.format(context=context, question=query))
    
    return response.content

# Test it
test_queries = [
    "What is the refund policy?",
    "How do I configure API authentication?",
    "What are the system requirements?",
    "How can I contact support?"
]

print("Testing RAG Pipeline with Multiple Queries:\n")
print("=" * 60)

for query in test_queries:
    answer = simple_rag_search(query, vector_store, llm)
    print(f"Q: {query}")
    print(f"A: {answer}\n")
    print("-" * 60)
